In [2]:
import pandas as pd
import glob
import os

def load_and_merge_trades(folder_path):
    """
    Ingests multiple daily trade Excel files and merges them into a single DataFrame.
    """
    print(f"Scanning directory: {folder_path}")
    
    # Locate all Excel files in the specified folder
    # Change to '*.xls' if they are strictly the older format
    file_pattern = os.path.join(folder_path, "*.xls*") 
    file_list = glob.glob(file_pattern)
    
    if not file_list:
        print("No Excel files found in the directory.")
        return pd.DataFrame()
    
    print(f"Found {len(file_list)} files. Beginning ingestion...")
    
    df_list = []
    for file in file_list:
        try:
            # Read each Excel file
            temp_df = pd.read_excel(file)
            
            # Optional: Add a column to track the source file (good for debugging)
            temp_df['Source_File'] = os.path.basename(file)
            
            df_list.append(temp_df)
            print(f"Successfully loaded: {os.path.basename(file)}")
            
        except Exception as e:
            print(f"Error loading {os.path.basename(file)}: {e}")
            
    # Concatenate all DataFrames into one master DataFrame
    master_trades_df = pd.concat(df_list, ignore_index=True)
    print(f"\nMerge complete. Master DataFrame shape: {master_trades_df.shape}")
    
    return master_trades_df

# Example usage: Replace with the actual path to your trades folder
TARGET_FOLDER = r'04 – Case SENTINEL\Trades'
combined_trades = load_and_merge_trades(TARGET_FOLDER)




Scanning directory: 04 – Case SENTINEL\Trades
Found 17 files. Beginning ingestion...
Successfully loaded: 554412_13_Apr_2007_trade.xls
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
Successfully loaded: 554412_17_July_2007.xls
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
Successfully loaded: 554412_18_July_2007.xls
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
Successfully loaded: 554412_19_July_2007.xls
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
Successfully loaded: 554412_20_July_2007.xls
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
Successfully loaded: 555835_13_APR_2007.xls
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
Successfully loaded: 555835_17_July_2007.xls
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
Successfully loaded: 555835_18_July_2007.xls
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
Success

In [3]:
# Define the exact path and filename for your CSV
csv_output_path = r'E:\Synapse-BI\04 – Case SENTINEL\master_trades_combined.csv'

# Export the DataFrame to CSV
# index=False ensures pandas doesn't write the row numbers as an extra unnamed column
combined_trades.to_csv(csv_output_path, index=False)

print(f"Successfully saved DataFrame to {csv_output_path}")

Successfully saved DataFrame to E:\Synapse-BI\04 – Case SENTINEL\master_trades_combined.csv


In [2]:
import pandas as pd
import networkx as nx

# 1. Load the data with low_memory=False to fix the DtypeWarning
df_trades = pd.read_csv(r'E:\Synapse-BI\04 – Case SENTINEL\master_trades_combined.csv', low_memory=False)

# 2. Clean up the inconsistent column names from the merged files
# Combine 'Sell Client Code' and 'Sell Client code' into a single reliable column
if 'Sell Client code' in df_trades.columns:
    df_trades['Sell Client Code'] = df_trades['Sell Client Code'].fillna(df_trades['Sell Client code'])

# Drop the redundant columns to keep memory clean
cols_to_drop = ['Sell Client code', 'Buy Timestamp', 'Sell Timestamp']
df_trades.drop(columns=[c for c in cols_to_drop if c in df_trades.columns], inplace=True)

# 3. Build the graphs
def build_surveillance_graphs(df_trades):
    """
    Constructs the bipartite Member-Client graph and the directed Client-Client graph.
    """
    B = nx.Graph()
    
    # Extract unique Buy-side and Sell-side Member-Client relationships
    # Using the correct 'BUY_MEMBER_CODE' and 'SELL_MEMBER_CODE'
    buy_links = df_trades[['BUY_MEMBER_CODE', 'Buy Client Code']].dropna().drop_duplicates()
    sell_links = df_trades[['SELL_MEMBER_CODE', 'Sell Client Code']].dropna().drop_duplicates()
    
    # Add nodes with bipartite attribute (0 for Members, 1 for Clients)
    members = set(buy_links['BUY_MEMBER_CODE']).union(set(sell_links['SELL_MEMBER_CODE']))
    clients = set(buy_links['Buy Client Code']).union(set(sell_links['Sell Client Code']))
    
    B.add_nodes_from(members, bipartite=0, type='Member')
    B.add_nodes_from(clients, bipartite=1, type='Client')
    
    # Add edges
    B.add_edges_from(buy_links.values)
    B.add_edges_from(sell_links.values)
    
    # ---------------------------------------------------------
    # GRAPH 2: Client -> Client Directed Trade Graph (Flow of Shares)
    # ---------------------------------------------------------
    DG = nx.DiGraph()
    
    # Aggregate trade quantity between counterparties
    # Source = Seller, Target = Buyer
    edge_weights = df_trades.groupby(['Sell Client Code', 'Buy Client Code'])['TRADE_QUANTITY'].sum().reset_index()
    
    # Add directed edges with quantity as weight
    for _, row in edge_weights.iterrows():
        DG.add_edge(row['Sell Client Code'], 
                    row['Buy Client Code'], 
                    weight=row['TRADE_QUANTITY'])
        
    return B, DG

# Run the function
B_graph, Client_DG = build_surveillance_graphs(df_trades)

print(f"Bipartite Graph: {B_graph.number_of_nodes()} nodes, {B_graph.number_of_edges()} edges")
print(f"Directed Graph: {Client_DG.number_of_nodes()} nodes, {Client_DG.number_of_edges()} edges")

Bipartite Graph: 37512 nodes, 37662 edges
Directed Graph: 36964 nodes, 173687 edges


In [3]:
import pandas as pd
import networkx as nx
from tqdm.auto import tqdm # Progress bar!

def ultra_fast_loop_detection(df_trades, max_length=3, min_volume_threshold=500):
    print(f"Starting ultra-optimized loop detection (Min Volume: {min_volume_threshold})...")
    all_suspicious_loops = []
    
    # Group by Stock and Day
    grouped = df_trades.groupby(['SCRIP_CODE', 'TRADE_DATE'])
    
    # Wrap 'grouped' in tqdm for a nice loading bar!
    for (scrip, date), group_df in tqdm(grouped, desc="Analyzing Scrip-Date Slices"):
        
        # 1. Aggregate edges
        edge_weights = group_df.groupby(['Sell Client Code', 'Buy Client Code'])['TRADE_QUANTITY'].sum().reset_index()
        
        # 2. PRUNE THE NOISE: Drop tiny retail trades that clog the algorithm
        edge_weights = edge_weights[edge_weights['TRADE_QUANTITY'] >= min_volume_threshold]
        
        # If no significant trades remain, skip to the next day
        if edge_weights.empty:
            continue
            
        # Build the mini-graph
        mini_DG = nx.DiGraph()
        for _, row in edge_weights.iterrows():
            mini_DG.add_edge(row['Sell Client Code'], row['Buy Client Code'], weight=row['TRADE_QUANTITY'])
            
        # 3. PRUNE DEAD ENDS: A loop requires buying AND selling. 
        # If someone only bought (out-degree 0) or only sold (in-degree 0), delete them.
        while True:
            dead_ends = [n for n in mini_DG.nodes if mini_DG.in_degree(n) == 0 or mini_DG.out_degree(n) == 0]
            if not dead_ends:
                break
            mini_DG.remove_nodes_from(dead_ends)
            
        # If the graph was entirely dead-ends, skip
        if len(mini_DG.nodes) == 0:
            continue
            
        # 4. Now run cycle detection on the highly-filtered, suspect-only graph
        cycles = list(nx.simple_cycles(mini_DG, length_bound=max_length))
        cycles = [c for c in cycles if len(c) > 1] 
        
        for cycle in cycles:
            total_vol = 0
            min_edge_vol = float('inf')
            
            for i in range(len(cycle)):
                u = cycle[i]
                v = cycle[(i + 1) % len(cycle)]
                weight = mini_DG[u][v]['weight']
                total_vol += weight
                if weight < min_edge_vol:
                    min_edge_vol = weight
                    
            all_suspicious_loops.append({
                'SCRIP_CODE': scrip,
                'TRADE_DATE': date,
                'Cluster_Nodes': str(cycle),
                'Loop_Length': len(cycle),
                'Total_Loop_Volume': total_vol,
                'Bottleneck_Volume': min_edge_vol
            })
            
    df_loops = pd.DataFrame(all_suspicious_loops)
    
    if not df_loops.empty:
        df_loops = df_loops.sort_values(by='Bottleneck_Volume', ascending=False).reset_index(drop=True)
        print(f"\nDone! Found {len(df_loops)} suspicious loops.")
    else:
        print("\nDone! No loops found matching these criteria.")
        
    return df_loops

# Run it! I've set the threshold to 500 shares to strip out the retail noise.
# If it's STILL slow, stop it and increase min_volume_threshold to 1000 or 5000.
optimized_loops_df = ultra_fast_loop_detection(df_trades, max_length=3, min_volume_threshold=500)
display(optimized_loops_df.head(10))

Starting ultra-optimized loop detection (Min Volume: 500)...


Analyzing Scrip-Date Slices:   0%|          | 0/11 [00:00<?, ?it/s]


Done! Found 1809 suspicious loops.


,SCRIP_CODE,TRADE_DATE,Cluster_Nodes,Loop_Length,Total_Loop_Volume,Bottleneck_Volume
0,556155,13-Apr-2007,"['L16WN', 'K17329']",2,180202,81062
1,556155,13-Apr-2007,"['L16WN', 'Y2B125']",2,130250,60397
2,556155,13-Apr-2007,"['L16WN', 'E23001']",2,125261,49609
3,556155,13-Apr-2007,"['L16WN', 'N14G230']",2,67459,33392
4,556155,13-Apr-2007,"['L16WN', '7120']",2,64918,30396
5,556155,13-Apr-2007,"['L16WN', '0398']",2,37795,18229
6,556155,13-Apr-2007,"['L16WN', '6092']",2,36479,18206
7,556155,13-Apr-2007,"['L16WN', 'Z1001']",2,35386,17641
8,556155,13-Apr-2007,"['L16WN', 'X3SL']",2,36456,16778
9,556155,13-Apr-2007,"['L16WN', '81025']",2,33726,14637


## Stage 1 — Task 1: Validate Suspected Entities
Cross-reference the provided suspect list against detected loops and raw trade microstructure to determine whether each flagged client has evidence of manipulative circular trading.

In [6]:
import ast
import numpy as np

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Explode detected loops → one row per (scrip, date, client)
# ─────────────────────────────────────────────────────────────────────────────
def explode_loops(df_loops):
    """Parse Cluster_Nodes strings and return one row per client in each loop."""
    records = []
    for _, row in df_loops.iterrows():
        try:
            nodes = ast.literal_eval(row['Cluster_Nodes'])
        except Exception:
            continue
        for client in nodes:
            records.append({
                'SCRIP_CODE'        : row['SCRIP_CODE'],
                'TRADE_DATE'        : row['TRADE_DATE'],
                'client_id'         : str(client).strip(),
                'Loop_Length'       : row['Loop_Length'],
                'Total_Loop_Volume' : row['Total_Loop_Volume'],
                'Bottleneck_Volume' : row['Bottleneck_Volume'],
            })
    return pd.DataFrame(records)

loops_exploded = explode_loops(optimized_loops_df)
print(f"Loop memberships resolved: {len(loops_exploded)} client-loop entries across "
      f"{loops_exploded['client_id'].nunique()} unique clients")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Per-suspect loop participation metrics
# ─────────────────────────────────────────────────────────────────────────────
suspects_df['client_id'] = suspects_df['client_id'].astype(str).str.strip()
suspects_df['scrip_code'] = suspects_df['scrip_code'].astype(str).str.strip()
loops_exploded['SCRIP_CODE'] = loops_exploded['SCRIP_CODE'].astype(str).str.strip()

# Aggregate loop stats per (scrip_code, client_id) — matching the suspect granularity
loop_stats = (
    loops_exploded
    .groupby(['SCRIP_CODE', 'client_id'])
    .agg(
        Loops_Participated   = ('Bottleneck_Volume', 'count'),
        Max_Loop_Volume      = ('Total_Loop_Volume', 'max'),
        Total_Bottleneck_Vol = ('Bottleneck_Volume', 'sum'),
        Avg_Loop_Length      = ('Loop_Length', 'mean'),
        Distinct_Dates       = ('TRADE_DATE', 'nunique'),
    )
    .reset_index()
    .rename(columns={'SCRIP_CODE': 'scrip_code'})
)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: Trade-level reciprocity for each suspected client
# ─────────────────────────────────────────────────────────────────────────────
df_trades['Buy Client Code'] = df_trades['Buy Client Code'].astype(str).str.strip()
df_trades['Sell Client Code'] = df_trades['Sell Client Code'].astype(str).str.strip()
df_trades['SCRIP_CODE'] = df_trades['SCRIP_CODE'].astype(str).str.strip()

buy_vol  = df_trades.groupby(['SCRIP_CODE', 'Buy Client Code'])['TRADE_QUANTITY'].sum().reset_index()
buy_vol.columns  = ['scrip_code', 'client_id', 'Total_Buy_Vol']

sell_vol = df_trades.groupby(['SCRIP_CODE', 'Sell Client Code'])['TRADE_QUANTITY'].sum().reset_index()
sell_vol.columns = ['scrip_code', 'client_id', 'Total_Sell_Vol']

trade_activity = pd.merge(buy_vol, sell_vol, on=['scrip_code', 'client_id'], how='outer').fillna(0)
trade_activity['Total_Traded_Vol'] = trade_activity['Total_Buy_Vol'] + trade_activity['Total_Sell_Vol']

# Reciprocity Ratio: 1.0 = perfectly balanced buys & sells (circular trading hallmark)
trade_activity['Reciprocity_Ratio'] = trade_activity.apply(
    lambda r: min(r['Total_Buy_Vol'], r['Total_Sell_Vol']) / max(r['Total_Buy_Vol'], r['Total_Sell_Vol'])
    if max(r['Total_Buy_Vol'], r['Total_Sell_Vol']) > 0 else 0, axis=1
)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: Counterparty diversity — how many distinct clients traded with this client?
# ─────────────────────────────────────────────────────────────────────────────
buy_cp  = df_trades.groupby(['SCRIP_CODE', 'Buy Client Code'])['Sell Client Code'].nunique().reset_index()
buy_cp.columns  = ['scrip_code', 'client_id', 'Distinct_Counterparties_as_Buyer']

sell_cp = df_trades.groupby(['SCRIP_CODE', 'Sell Client Code'])['Buy Client Code'].nunique().reset_index()
sell_cp.columns = ['scrip_code', 'client_id', 'Distinct_Counterparties_as_Seller']

cp_stats = pd.merge(buy_cp, sell_cp, on=['scrip_code', 'client_id'], how='outer').fillna(0)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: Merge all evidence onto the suspects list
# ─────────────────────────────────────────────────────────────────────────────
validation = suspects_df.copy()
validation = validation.merge(loop_stats,     on=['scrip_code', 'client_id'], how='left')
validation = validation.merge(trade_activity, on=['scrip_code', 'client_id'], how='left')
validation = validation.merge(cp_stats,       on=['scrip_code', 'client_id'], how='left')
validation.fillna({'Loops_Participated': 0, 'Max_Loop_Volume': 0,
                   'Total_Bottleneck_Vol': 0, 'Avg_Loop_Length': 0,
                   'Distinct_Dates': 0, 'Total_Buy_Vol': 0,
                   'Total_Sell_Vol': 0, 'Total_Traded_Vol': 0,
                   'Reciprocity_Ratio': 0, 'Distinct_Counterparties_as_Buyer': 0,
                   'Distinct_Counterparties_as_Seller': 0}, inplace=True)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 6: Classification
# ─────────────────────────────────────────────────────────────────────────────
def classify_entity(row):
    in_loops     = row['Loops_Participated'] > 0
    high_recipr  = row['Reciprocity_Ratio'] >= 0.70
    multi_date   = row['Distinct_Dates'] >= 2
    high_vol     = row['Total_Bottleneck_Vol'] > 5000

    if in_loops and high_recipr and multi_date:
        return 'Circular Trading'
    elif in_loops and high_recipr:
        return 'Circular Trading'
    elif in_loops and not high_recipr:
        return 'Possible Circular / Pump-Dump Coordination'
    elif high_recipr and row['Total_Traded_Vol'] > 0:
        return 'Reciprocal — Insufficient Loop Evidence'
    elif row['Total_Traded_Vol'] > 0:
        return 'Legitimate High-Activity Behavior'
    else:
        return 'Insufficient Evidence'

validation['Classification'] = validation.apply(classify_entity, axis=1)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 7: Summary Report
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 65)
print("STAGE 1 — TASK 1: ENTITY VALIDATION SUMMARY")
print("=" * 65)
print(f"Total suspects assessed : {len(validation)}")
print(f"Suspects validated      : {(validation['Loops_Participated'] > 0).sum()} (found in circular loops)")
print(f"Suspects NOT in loops   : {(validation['Loops_Participated'] == 0).sum()} (no loop evidence)")
print()
print("Classification breakdown:")
print(validation['Classification'].value_counts().to_string())
print()
print("Top 20 highest-risk validated suspects (by Total Bottleneck Volume):")
top_suspects = (
    validation[validation['Loops_Participated'] > 0]
    .sort_values('Total_Bottleneck_Vol', ascending=False)
    .head(20)
)
display(top_suspects[['scrip_code', 'member_code', 'client_id',
                       'Loops_Participated', 'Distinct_Dates',
                       'Total_Bottleneck_Vol', 'Reciprocity_Ratio',
                       'Classification']].reset_index(drop=True))

# Suspects on list but with NO evidence
no_evidence = validation[validation['Loops_Participated'] == 0].copy()
print(f"\nSuspects with NO circular loop evidence ({len(no_evidence)} entities) — potential false positives:")
display(no_evidence[['scrip_code', 'member_code', 'client_id',
                      'Total_Traded_Vol', 'Reciprocity_Ratio', 'Classification']].reset_index(drop=True))


Loop memberships resolved: 4637 client-loop entries across 868 unique clients
STAGE 1 — TASK 1: ENTITY VALIDATION SUMMARY
Total suspects assessed : 2212
Suspects validated      : 575 (found in circular loops)
Suspects NOT in loops   : 1637 (no loop evidence)

Classification breakdown:
Classification
Insufficient Evidence                         844
Reciprocal — Insufficient Loop Evidence       764
Circular Trading                              570
Legitimate High-Activity Behavior              29
Possible Circular / Pump-Dump Coordination      5

Top 20 highest-risk validated suspects (by Total Bottleneck Volume):


,scrip_code,member_code,client_id,Loops_Participated,Distinct_Dates,Total_Bottleneck_Vol,Reciprocity_Ratio,Classification
0,556155,927,L16WN,1568.0,2.0,2171489.0,0.99181,Circular Trading
1,556155,3175,L16WN,1568.0,2.0,2171489.0,0.99181,Circular Trading
2,556155,366,L16WN,1568.0,2.0,2171489.0,0.99181,Circular Trading
3,556155,770,L16WN,1568.0,2.0,2171489.0,0.99181,Circular Trading
4,556155,636,L16WN,1568.0,2.0,2171489.0,0.99181,Circular Trading
5,556155,167,L16WN,1568.0,2.0,2171489.0,0.99181,Circular Trading
6,556155,811,L16WN,1568.0,2.0,2171489.0,0.99181,Circular Trading
7,556155,498,L16WN,1568.0,2.0,2171489.0,0.99181,Circular Trading
8,556155,617,L16WN,1568.0,2.0,2171489.0,0.99181,Circular Trading
9,556155,3056,L16WN,1568.0,2.0,2171489.0,0.99181,Circular Trading



Suspects with NO circular loop evidence (1637 entities) — potential false positives:


,scrip_code,member_code,client_id,Total_Traded_Vol,Reciprocity_Ratio,Classification
0,554412,212,65004,5000.0,1.000000,Reciprocal — Insufficient Loop Evidence
1,555835,112,S8K200,1000.0,1.000000,Reciprocal — Insufficient Loop Evidence
2,555835,202,W4EEPKAND,4000.0,1.000000,Reciprocal — Insufficient Loop Evidence
3,555835,212,491075,2000.0,1.000000,Reciprocal — Insufficient Loop Evidence
4,555835,271,10S074,3400.0,1.000000,Reciprocal — Insufficient Loop Evidence
...,...,...,...,...,...,...
1632,556155,802,S8OJ12,0.0,0.000000,Insufficient Evidence
1633,556155,933,E2322,15322.0,0.861725,Reciprocal — Insufficient Loop Evidence
1634,556155,956,420,200.0,1.000000,Reciprocal — Insufficient Loop Evidence
1635,556155,960,46002,0.0,0.000000,Insufficient Evidence


## Stage 1 — Task 2: Classify Mechanism
Each validated entity or cluster is classified as one of:
- **Circular Trading** — closed loops with high reciprocity, no economic rationale
- **Pump & Dump Coordination** — coordinated volume surge with upward price movement
- **Infrastructure-Linked Coordination** — shared member, location, or trader IDs
- **Legitimate High-Activity Behavior** — high turnover, low reciprocity, no persistent loops
- **Insufficient Evidence**

In [7]:
import numpy as np

# ─────────────────────────────────────────────────────────────────────────────
# SIGNAL A: Pump & Dump Detection
# For each (SCRIP_CODE, TRADE_DATE): measure price drift during high-volume periods.
# P&D hallmark = volume spike + monotonic price rise (buying pressure), then drop.
# ─────────────────────────────────────────────────────────────────────────────

# Sort trades by scrip, date, then time
df_trades['TRADE_TIME'] = pd.to_datetime(df_trades['TRADE_TIME'], errors='coerce')
df_sorted = df_trades.dropna(subset=['TRADE_TIME']).sort_values(['SCRIP_CODE', 'TRADE_DATE', 'TRADE_TIME'])

def price_drift(group):
    """Returns (pct_price_change, is_monotonic_up, volume) for a scrip-date block."""
    rates = group['TRADE_RATE'].dropna().values
    if len(rates) < 3:
        return 0.0, False, group['TRADE_QUANTITY'].sum()
    pct_change = (rates[-1] - rates[0]) / rates[0] * 100
    # Monotonic up: more than 60% of consecutive differences are positive
    diffs = np.diff(rates)
    mono_up = (diffs > 0).mean() > 0.60
    return round(pct_change, 4), bool(mono_up), group['TRADE_QUANTITY'].sum()

price_signals = (
    df_sorted.groupby(['SCRIP_CODE', 'TRADE_DATE'])
    .apply(lambda g: pd.Series(price_drift(g), index=['Price_Pct_Change', 'Monotonic_Up', 'Daily_Volume']))
    .reset_index()
)
price_signals['SCRIP_CODE'] = price_signals['SCRIP_CODE'].astype(str)

print("Price signals per scrip-date:")
display(price_signals.sort_values('Price_Pct_Change', ascending=False))

# ─────────────────────────────────────────────────────────────────────────────
# SIGNAL B: Infrastructure-Linked Coordination
# Clients sharing Member Code, Location ID, or Trader ID within a loop cluster.
# ─────────────────────────────────────────────────────────────────────────────

# Build a per-client infrastructure fingerprint
buy_infra = df_trades[['Buy Client Code', 'SCRIP_CODE', 'BUY_MEMBER_CODE', 'BUY_LOCATION_ID', 'BUY_TRADER_ID']].copy()
buy_infra.columns = ['client_id', 'scrip_code', 'MEMBER_CODE', 'LOCATION_ID', 'TRADER_ID']

sell_infra = df_trades[['Sell Client Code', 'SCRIP_CODE', 'SELL_MEMBER_CODE', 'SELL_LOCATION_ID', 'SELL_TRADER_ID']].copy()
sell_infra.columns = ['client_id', 'scrip_code', 'MEMBER_CODE', 'LOCATION_ID', 'TRADER_ID']

infra_df = pd.concat([buy_infra, sell_infra], ignore_index=True)
infra_df['client_id'] = infra_df['client_id'].astype(str).str.strip()
infra_df['scrip_code'] = infra_df['scrip_code'].astype(str).str.strip()

# Mode infrastructure per (client, scrip)
infra_summary = (
    infra_df.groupby(['scrip_code', 'client_id'])
    .agg(
        Primary_Member   = ('MEMBER_CODE',  lambda x: x.mode()[0] if len(x.mode()) > 0 else np.nan),
        Primary_Location = ('LOCATION_ID',  lambda x: x.mode()[0] if len(x.mode()) > 0 else np.nan),
        Primary_Trader   = ('TRADER_ID',    lambda x: x.mode()[0] if len(x.mode()) > 0 else np.nan),
    )
    .reset_index()
)

# ─────────────────────────────────────────────────────────────────────────────
# SIGNAL C: Per-loop cluster infrastructure overlap
# Within each loop, do participants share Member / Location / Trader?
# ─────────────────────────────────────────────────────────────────────────────

import ast

def cluster_infra_overlap(row, infra_summary):
    try:
        nodes = [str(n).strip() for n in ast.literal_eval(row['Cluster_Nodes'])]
    except Exception:
        return False, False, False
    scrip = str(row['SCRIP_CODE']).strip()
    members = infra_summary[(infra_summary['scrip_code'] == scrip) & 
                             (infra_summary['client_id'].isin(nodes))]
    shared_member   = members['Primary_Member'].nunique()   <= 1 and len(members) > 1
    shared_location = members['Primary_Location'].nunique() <= 1 and len(members) > 1
    shared_trader   = members['Primary_Trader'].nunique()   <= 1 and len(members) > 1
    return shared_member, shared_location, shared_trader

overlap_results = optimized_loops_df.apply(
    lambda r: pd.Series(cluster_infra_overlap(r, infra_summary),
                        index=['Shared_Member', 'Shared_Location', 'Shared_Trader']),
    axis=1
)
optimized_loops_df = pd.concat([optimized_loops_df, overlap_results], axis=1)
optimized_loops_df['Infra_Linked'] = (
    optimized_loops_df['Shared_Member'] | 
    optimized_loops_df['Shared_Location'] | 
    optimized_loops_df['Shared_Trader']
)

print(f"\nLoops with infrastructure linkage: {optimized_loops_df['Infra_Linked'].sum()} "
      f"of {len(optimized_loops_df)}")

# ─────────────────────────────────────────────────────────────────────────────
# FINAL MECHANISM CLASSIFICATION
# Merge all signals onto the validation table and assign final mechanism label.
# ─────────────────────────────────────────────────────────────────────────────

# Attach price signals to loops
optimized_loops_df['SCRIP_CODE'] = optimized_loops_df['SCRIP_CODE'].astype(str)
price_signals['SCRIP_CODE'] = price_signals['SCRIP_CODE'].astype(str)
loops_with_price = optimized_loops_df.merge(
    price_signals[['SCRIP_CODE', 'TRADE_DATE', 'Price_Pct_Change', 'Monotonic_Up']],
    on=['SCRIP_CODE', 'TRADE_DATE'], how='left'
)

# Re-explode to client level with full signals
def explode_loops_full(df_loops):
    records = []
    for _, row in df_loops.iterrows():
        try:
            nodes = ast.literal_eval(row['Cluster_Nodes'])
        except Exception:
            continue
        for client in nodes:
            records.append({
                'SCRIP_CODE'        : str(row['SCRIP_CODE']).strip(),
                'TRADE_DATE'        : row['TRADE_DATE'],
                'client_id'         : str(client).strip(),
                'Loop_Length'       : row['Loop_Length'],
                'Total_Loop_Volume' : row['Total_Loop_Volume'],
                'Bottleneck_Volume' : row['Bottleneck_Volume'],
                'Infra_Linked'      : row['Infra_Linked'],
                'Price_Pct_Change'  : row.get('Price_Pct_Change', 0),
                'Monotonic_Up'      : row.get('Monotonic_Up', False),
            })
    return pd.DataFrame(records)

loops_full = explode_loops_full(loops_with_price)

# Aggregate per (scrip, client) — take worst-case signals
loop_signals = (
    loops_full.groupby(['SCRIP_CODE', 'client_id'])
    .agg(
        Loops_Participated   = ('Bottleneck_Volume', 'count'),
        Max_Bottleneck       = ('Bottleneck_Volume', 'max'),
        Total_Bottleneck_Vol = ('Bottleneck_Volume', 'sum'),
        Avg_Loop_Length      = ('Loop_Length', 'mean'),
        Distinct_Dates       = ('TRADE_DATE', 'nunique'),
        Any_Infra_Linked     = ('Infra_Linked', 'max'),
        Max_Price_Change     = ('Price_Pct_Change', 'max'),
        Any_Monotonic_Up     = ('Monotonic_Up', 'max'),
    )
    .reset_index()
    .rename(columns={'SCRIP_CODE': 'scrip_code'})
)

# Rebuild full validation with refined signals
validation2 = suspects_df.copy()
validation2 = validation2.merge(loop_signals,     on=['scrip_code', 'client_id'], how='left')
validation2 = validation2.merge(trade_activity,   on=['scrip_code', 'client_id'], how='left')
validation2 = validation2.merge(infra_summary,    on=['scrip_code', 'client_id'], how='left')
validation2.fillna({
    'Loops_Participated': 0, 'Max_Bottleneck': 0, 'Total_Bottleneck_Vol': 0,
    'Avg_Loop_Length': 0, 'Distinct_Dates': 0, 'Any_Infra_Linked': False,
    'Max_Price_Change': 0, 'Any_Monotonic_Up': False,
    'Total_Buy_Vol': 0, 'Total_Sell_Vol': 0,
    'Total_Traded_Vol': 0, 'Reciprocity_Ratio': 0,
}, inplace=True)

def classify_mechanism(row):
    in_loops    = row['Loops_Participated'] > 0
    high_recip  = row['Reciprocity_Ratio'] >= 0.70
    infra       = row.get('Any_Infra_Linked', False)
    pump        = row['Max_Price_Change'] > 3.0 and row.get('Any_Monotonic_Up', False)
    multi_date  = row['Distinct_Dates'] >= 2

    if in_loops and infra:
        return 'Infrastructure-Linked Coordination'
    elif in_loops and pump:
        return 'Pump & Dump Coordination'
    elif in_loops and high_recip:
        return 'Circular Trading'
    elif in_loops:
        return 'Circular Trading'          # loop presence is sufficient
    elif high_recip and row['Total_Traded_Vol'] > 0:
        return 'Reciprocal — Insufficient Loop Evidence'
    elif row['Total_Traded_Vol'] > 0:
        return 'Legitimate High-Activity Behavior'
    else:
        return 'Insufficient Evidence'

validation2['Mechanism'] = validation2.apply(classify_mechanism, axis=1)

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("STAGE 1 — TASK 2: MECHANISM CLASSIFICATION SUMMARY")
print("=" * 65)
print(f"Total suspects classified: {len(validation2)}\n")
print("Mechanism breakdown:")
mech_counts = validation2['Mechanism'].value_counts()
print(mech_counts.to_string())

print("\nInfrastructure-Linked loops detail (sample top 10):")
infra_loops = optimized_loops_df[optimized_loops_df['Infra_Linked']].sort_values('Bottleneck_Volume', ascending=False)
display(infra_loops[['SCRIP_CODE', 'TRADE_DATE', 'Cluster_Nodes', 'Loop_Length',
                      'Bottleneck_Volume', 'Shared_Member', 'Shared_Location', 'Shared_Trader']].head(10).reset_index(drop=True))

print("\nScrip-level price signals (Pump & Dump indicator):")
display(price_signals.sort_values('Price_Pct_Change', ascending=False).reset_index(drop=True))

print("\nTop 15 Circular Trading suspects:")
ct = validation2[validation2['Mechanism'] == 'Circular Trading'].sort_values('Total_Bottleneck_Vol', ascending=False).head(15)
display(ct[['scrip_code', 'member_code', 'client_id', 'Loops_Participated',
             'Distinct_Dates', 'Total_Bottleneck_Vol', 'Reciprocity_Ratio', 'Mechanism']].reset_index(drop=True))

print("\nTop 15 Pump & Dump suspects:")
pd_suspects = validation2[validation2['Mechanism'] == 'Pump & Dump Coordination'].sort_values('Total_Bottleneck_Vol', ascending=False).head(15)
if pd_suspects.empty:
    print("  None meeting threshold. Check price_signals for scrip-level evidence above.")
else:
    display(pd_suspects[['scrip_code', 'member_code', 'client_id', 'Max_Price_Change',
                          'Total_Bottleneck_Vol', 'Mechanism']].reset_index(drop=True))


E:\Temp\ipykernel_21436\3583964730.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_trades['TRADE_TIME'] = pd.to_datetime(df_trades['TRADE_TIME'], errors='coerce')


Price signals per scrip-date:


E:\Temp\ipykernel_21436\3583964730.py:26: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series(price_drift(g), index=['Price_Pct_Change', 'Monotonic_Up', 'Daily_Volume']))


,SCRIP_CODE,TRADE_DATE,Price_Pct_Change,Monotonic_Up,Daily_Volume
9,556155,13-Apr-2007,51.9238,False,12426046
4,554412,20-Jul-2007,5.2778,False,608867
7,555835,18-Jul-2007,1.9615,False,152556
5,555835,13-Apr-2007,1.1278,False,295163
8,555835,19-Jul-2007,-0.6028,False,41279
2,554412,18-Jul-2007,-1.6379,False,109849
6,555835,17-Jul-2007,-2.1908,False,79689
10,556155,17-Jul-2007,-2.2644,False,263626
3,554412,19-Jul-2007,-2.3963,False,79575
0,554412,13-Apr-2007,-3.8988,False,32093



Loops with infrastructure linkage: 3 of 1809

STAGE 1 — TASK 2: MECHANISM CLASSIFICATION SUMMARY
Total suspects classified: 2212

Mechanism breakdown:
Mechanism
Insufficient Evidence                      844
Reciprocal — Insufficient Loop Evidence    764
Circular Trading                           490
Infrastructure-Linked Coordination          85
Legitimate High-Activity Behavior           29

Infrastructure-Linked loops detail (sample top 10):


...


Scrip-level price signals (Pump & Dump indicator):


,SCRIP_CODE,TRADE_DATE,Price_Pct_Change,Monotonic_Up,Daily_Volume
0,556155,13-Apr-2007,51.9238,False,12426046
1,554412,20-Jul-2007,5.2778,False,608867
2,555835,18-Jul-2007,1.9615,False,152556
3,555835,13-Apr-2007,1.1278,False,295163
4,555835,19-Jul-2007,-0.6028,False,41279
5,554412,18-Jul-2007,-1.6379,False,109849
6,555835,17-Jul-2007,-2.1908,False,79689
7,556155,17-Jul-2007,-2.2644,False,263626
8,554412,19-Jul-2007,-2.3963,False,79575
9,554412,13-Apr-2007,-3.8988,False,32093



Top 15 Circular Trading suspects:


,scrip_code,member_code,client_id,Loops_Participated,Distinct_Dates,Total_Bottleneck_Vol,Reciprocity_Ratio,Mechanism
0,556155,318,K17329,249.0,2.0,340952.0,1.000000,Circular Trading
1,556155,120,E23001,211.0,1.0,261036.0,0.992977,Circular Trading
2,556155,469,Y2B125,116.0,1.0,174402.0,0.999962,Circular Trading
3,556155,3060,N14G230,44.0,1.0,74540.0,0.967897,Circular Trading
4,556155,97,7120,52.0,1.0,73338.0,0.979474,Circular Trading
5,556155,97,7120,52.0,1.0,73338.0,0.979474,Circular Trading
6,556155,97,7120,52.0,1.0,73338.0,0.979474,Circular Trading
7,556155,562,G21119,36.0,1.0,52375.0,1.000000,Circular Trading
8,556155,3057,X3SL,43.0,1.0,46692.0,1.000000,Circular Trading
9,556155,792,3012,24.0,1.0,43444.0,1.000000,Circular Trading



Top 15 Pump & Dump suspects:
  None meeting threshold. Check price_signals for scrip-level evidence above.


## Stage 1 — Task 3: Network Analysis
Mandatory components per guidelines:
1. **Client ↔ Client trade graph** — already built (`Client_DG`); extend with reciprocity metrics
2. **Member ↔ Client linkage** — broker concentration & routing patterns
3. **Reciprocity analysis** — edge-level mutual trade detection
4. **Loop detection** — already complete (`optimized_loops_df`); summarise motif stats
5. **Synchronization patterns** — time-window co-trading between loop participants

In [9]:
import networkx as nx
import numpy as np
import ast

# ─────────────────────────────────────────────────────────────────────────────
# 3A: CLIENT ↔ CLIENT GRAPH — Reciprocity Analysis
# For each directed edge A→B, check if the reverse edge B→A also exists.
# Mutual edges = potential circular or wash trading pairs.
# ─────────────────────────────────────────────────────────────────────────────

reciprocal_pairs = []
for u, v, data in Client_DG.edges(data=True):
    if Client_DG.has_edge(v, u):
        w_fwd = data['weight']
        w_rev = Client_DG[v][u]['weight']
        min_vol   = min(w_fwd, w_rev)
        balance   = min_vol / max(w_fwd, w_rev)   # 1.0 = perfectly balanced
        reciprocal_pairs.append({
            'Client_A': u, 'Client_B': v,
            'Vol_A_to_B': w_fwd,
            'Vol_B_to_A': w_rev,
            'Shared_Vol': min_vol,
            'Balance_Ratio': round(balance, 4),
        })

reciprocal_df = pd.DataFrame(reciprocal_pairs)

# De-duplicate (A↔B and B↔A are the same pair)
reciprocal_df['pair'] = reciprocal_df.apply(
    lambda r: tuple(sorted([r['Client_A'], r['Client_B']])), axis=1)
reciprocal_df = reciprocal_df.drop_duplicates('pair').drop(columns='pair').reset_index(drop=True)

print(f"Total reciprocal (mutual) trading pairs : {len(reciprocal_df):,}")
print(f"Highly balanced pairs (ratio ≥ 0.90)    : {(reciprocal_df['Balance_Ratio'] >= 0.90).sum():,}")
print(f"Perfectly balanced pairs (ratio = 1.0)  : {(reciprocal_df['Balance_Ratio'] == 1.0).sum():,}")

print("\nTop 20 highest-volume reciprocal pairs:")
display(reciprocal_df.sort_values('Shared_Vol', ascending=False).head(20).reset_index(drop=True))

# ─────────────────────────────────────────────────────────────────────────────
# 3B: MEMBER ↔ CLIENT LINKAGE — Broker Concentration & Routing Patterns
# How many clients does each member route? Concentrated routing = risk signal.
# ─────────────────────────────────────────────────────────────────────────────

buy_mc  = df_trades[['BUY_MEMBER_CODE',  'Buy Client Code']].rename(
    columns={'BUY_MEMBER_CODE':'member_code', 'Buy Client Code':'client_id'})
sell_mc = df_trades[['SELL_MEMBER_CODE', 'Sell Client Code']].rename(
    columns={'SELL_MEMBER_CODE':'member_code', 'Sell Client Code':'client_id'})

member_client_map = pd.concat([buy_mc, sell_mc], ignore_index=True).drop_duplicates()
member_client_map['member_code'] = member_client_map['member_code'].astype(str).str.strip()
member_client_map['client_id']   = member_client_map['client_id'].astype(str).str.strip()

# How many distinct clients each member routes
member_stats = (
    member_client_map
    .groupby('member_code')['client_id']
    .nunique()
    .reset_index()
    .rename(columns={'client_id': 'Distinct_Clients'})
    .sort_values('Distinct_Clients', ascending=False)
)

# Volume per member — cast to str to match member_stats key
buy_vol_m  = df_trades.groupby('BUY_MEMBER_CODE')['TRADE_QUANTITY'].sum().reset_index().rename(
    columns={'BUY_MEMBER_CODE':'member_code','TRADE_QUANTITY':'Buy_Vol'})
buy_vol_m['member_code'] = buy_vol_m['member_code'].astype(str).str.strip()

sell_vol_m = df_trades.groupby('SELL_MEMBER_CODE')['TRADE_QUANTITY'].sum().reset_index().rename(
    columns={'SELL_MEMBER_CODE':'member_code','TRADE_QUANTITY':'Sell_Vol'})
sell_vol_m['member_code'] = sell_vol_m['member_code'].astype(str).str.strip()

member_stats = member_stats.merge(buy_vol_m,  on='member_code', how='left')
member_stats = member_stats.merge(sell_vol_m, on='member_code', how='left')
member_stats.fillna(0, inplace=True)
member_stats['Total_Vol'] = member_stats['Buy_Vol'] + member_stats['Sell_Vol']

# Proportion of loop-involved clients routed by each member
loop_clients = set(loops_full['client_id'].unique())
loop_client_per_member = (
    member_client_map[member_client_map['client_id'].isin(loop_clients)]
    .groupby('member_code')['client_id']
    .nunique()
    .reset_index()
    .rename(columns={'client_id': 'Loop_Clients_Routed'})
)
member_stats = member_stats.merge(loop_client_per_member, on='member_code', how='left')
member_stats['Loop_Clients_Routed'] = member_stats['Loop_Clients_Routed'].fillna(0).astype(int)
member_stats['Loop_Client_Pct'] = (
    member_stats['Loop_Clients_Routed'] / member_stats['Distinct_Clients'] * 100
).round(1)

print("\n" + "=" * 65)
print("3B: MEMBER ↔ CLIENT LINKAGE")
print("=" * 65)
print(f"Total unique members : {len(member_stats)}")
print("\nTop 20 members by loop-client concentration:")
display(member_stats.sort_values('Loop_Client_Pct', ascending=False).head(20).reset_index(drop=True))

# ─────────────────────────────────────────────────────────────────────────────
# 3C: LOOP DETECTION SUMMARY — Motif Statistics
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 65)
print("3C: LOOP DETECTION — MOTIF STATISTICS")
print("=" * 65)
print(f"Total suspicious loops found    : {len(optimized_loops_df):,}")
print(f"  Length-2 loops (A↔B pairs)    : {(optimized_loops_df['Loop_Length']==2).sum():,}")
print(f"  Length-3 loops (triangles)    : {(optimized_loops_df['Loop_Length']==3).sum():,}")
print(f"Infrastructure-linked loops     : {optimized_loops_df['Infra_Linked'].sum():,}")
print(f"Scrip-dates with loops          : {optimized_loops_df.groupby(['SCRIP_CODE','TRADE_DATE']).ngroups:,}")

per_scrip = optimized_loops_df.groupby('SCRIP_CODE').agg(
    Total_Loops      = ('Loop_Length','count'),
    Total_Bottleneck = ('Bottleneck_Volume','sum'),
    Avg_Bottleneck   = ('Bottleneck_Volume','mean'),
    Dates_Active     = ('TRADE_DATE','nunique'),
).reset_index().sort_values('Total_Bottleneck', ascending=False)
print("\nLoop distribution by scrip:")
display(per_scrip.reset_index(drop=True))

# ─────────────────────────────────────────────────────────────────────────────
# 3D: SYNCHRONIZATION PATTERNS — Time-Window Co-Trading
# For confirmed loop clusters: how tightly are trades synchronized?
# Measure: median inter-trade gap (in seconds) between loop participants.
# ─────────────────────────────────────────────────────────────────────────────

df_trades['TRADE_TIME_DT'] = pd.to_datetime(df_trades['TRADE_TIME'], errors='coerce')

sync_results = []

for _, loop_row in optimized_loops_df.iterrows():
    try:
        nodes = [str(n).strip() for n in ast.literal_eval(loop_row['Cluster_Nodes'])]
    except Exception:
        continue

    scrip = str(loop_row['SCRIP_CODE']).strip()
    date  = loop_row['TRADE_DATE']

    # All trades by loop-cluster clients on that scrip-date
    mask = (
        (df_trades['SCRIP_CODE'].astype(str).str.strip() == scrip) &
        (df_trades['TRADE_DATE'] == date) &
        (
            df_trades['Buy Client Code'].isin(nodes) |
            df_trades['Sell Client Code'].isin(nodes)
        )
    )
    cluster_trades = df_trades[mask].dropna(subset=['TRADE_TIME_DT']).sort_values('TRADE_TIME_DT')

    if len(cluster_trades) < 2:
        continue

    times = cluster_trades['TRADE_TIME_DT'].values
    gaps  = np.diff(times.astype('int64')) / 1e9  # nanoseconds → seconds
    gaps  = gaps[gaps >= 0]

    if len(gaps) == 0:
        continue

    sync_results.append({
        'SCRIP_CODE'        : scrip,
        'TRADE_DATE'        : date,
        'Cluster_Nodes'     : loop_row['Cluster_Nodes'],
        'Bottleneck_Volume' : loop_row['Bottleneck_Volume'],
        'Trades_in_Window'  : len(cluster_trades),
        'Median_Gap_Sec'    : round(float(np.median(gaps)), 2),
        'Min_Gap_Sec'       : round(float(np.min(gaps)), 2),
        'Max_Gap_Sec'       : round(float(np.max(gaps)), 2),
    })

sync_df = pd.DataFrame(sync_results)

if not sync_df.empty:
    print("\n" + "=" * 65)
    print("3D: SYNCHRONIZATION PATTERNS — TIME-WINDOW CO-TRADING")
    print("=" * 65)
    print(f"Loops with timestamp data : {len(sync_df):,}")
    print(f"Highly synchronized loops (median gap < 60s) : "
          f"{(sync_df['Median_Gap_Sec'] < 60).sum():,}")
    print("\nTop 20 most tightly synchronized loops (lowest median gap):")
    display(sync_df.sort_values('Median_Gap_Sec').head(20).reset_index(drop=True))
else:
    print("\n3D: No timestamp data available for synchronization analysis.")

# ─────────────────────────────────────────────────────────────────────────────
# NETWORK SUMMARY TABLE
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("TASK 3: NETWORK ANALYSIS — CONSOLIDATED SUMMARY")
print("=" * 65)
print(f"Client–Client graph  : {Client_DG.number_of_nodes():,} nodes | {Client_DG.number_of_edges():,} directed edges")
print(f"Member–Client graph  : {B_graph.number_of_nodes():,} nodes  | {B_graph.number_of_edges():,} edges")
print(f"Reciprocal pairs     : {len(reciprocal_df):,}  (balanced ≥0.9: {(reciprocal_df['Balance_Ratio']>=0.90).sum():,})")
print(f"Circular loops       : {len(optimized_loops_df):,}  (infra-linked: {optimized_loops_df['Infra_Linked'].sum():,})")
print(f"Members routing loop clients : {(member_stats['Loop_Clients_Routed']>0).sum():,} of {len(member_stats):,}")
if not sync_df.empty:
    print(f"Highly synced loops  : {(sync_df['Median_Gap_Sec']<60).sum():,}")


Total reciprocal (mutual) trading pairs : 14,691
Highly balanced pairs (ratio ≥ 0.90)    : 3,413
Perfectly balanced pairs (ratio = 1.0)  : 2,567

Top 20 highest-volume reciprocal pairs:


,Client_A,Client_B,Vol_A_to_B,Vol_B_to_A,Shared_Vol,Balance_Ratio
0,L16WN,L16WN,1132362,1132362,1132362,1.0000
1,L16WN,K17329,84414,99693,84414,0.8467
2,L16WN,Y2B125,69853,60397,60397,0.8646
3,L16WN,E23001,49609,75652,49609,0.6558
4,L16WN,N14G230,34067,33392,33392,0.9802
5,L16WN,7120,35537,30451,30451,0.8569
6,L16WN,0398,19566,18229,18229,0.9317
7,L16WN,6092,18273,18206,18206,0.9963
8,L16WN,Z1001,17745,17641,17641,0.9941
9,L16WN,X3SL,19678,16778,16778,0.8526



3B: MEMBER ↔ CLIENT LINKAGE
Total unique members : 548

Top 20 members by loop-client concentration:


,member_code,Distinct_Clients,Buy_Vol,Sell_Vol,Total_Vol,Loop_Clients_Routed,Loop_Client_Pct
0,730,1,40962.0,40962.0,81924.0,1,100.0
1,543,1,25430.0,25430.0,50860.0,1,100.0
2,379,1,22107.0,22095.0,44202.0,1,100.0
3,3056,1,37785.0,37785.0,75570.0,1,100.0
4,404,1,6000.0,6000.0,12000.0,1,100.0
5,478,1,44.0,44.0,88.0,1,100.0
6,734,1,1867.0,1867.0,3734.0,1,100.0
7,73,1,2547.0,2547.0,5094.0,1,100.0
8,100,1,3900.0,3900.0,7800.0,1,100.0
9,792,1,40000.0,40000.0,80000.0,1,100.0



3C: LOOP DETECTION — MOTIF STATISTICS
Total suspicious loops found    : 1,809
  Length-2 loops (A↔B pairs)    : 790
  Length-3 loops (triangles)    : 1,019
Infrastructure-linked loops     : 3
Scrip-dates with loops          : 5

Loop distribution by scrip:


,SCRIP_CODE,Total_Loops,Total_Bottleneck,Avg_Bottleneck,Dates_Active
0,556155,1774,2335572,1316.556933,2
1,555835,35,24791,708.314286,3



3D: SYNCHRONIZATION PATTERNS — TIME-WINDOW CO-TRADING
Loops with timestamp data : 1,809
Highly synchronized loops (median gap < 60s) : 1,809

Top 20 most tightly synchronized loops (lowest median gap):


,SCRIP_CODE,TRADE_DATE,Cluster_Nodes,Bottleneck_Volume,Trades_in_Window,Median_Gap_Sec,Min_Gap_Sec,Max_Gap_Sec
0,556155,13-Apr-2007,"['L16WN', 'Z114830', 'K17329']",500,112872,0.0,0.0,613.77
1,556155,13-Apr-2007,"['L16WN', '50P07']",7833,106502,0.0,0.0,613.77
2,556155,13-Apr-2007,"['0399', 'L16WN']",7428,106290,0.0,0.0,613.77
3,556155,13-Apr-2007,"['L16WN', 'M15612']",7374,106265,0.0,0.0,613.77
4,556155,13-Apr-2007,"['Y2AM199', 'L16WN']",7159,106023,0.0,0.0,613.77
5,556155,13-Apr-2007,"['M15S4777', 'L16WN']",6781,105982,0.0,0.0,613.77
6,556155,13-Apr-2007,"['Y2110656', 'L16WN']",6640,106076,0.0,0.0,613.77
7,556155,13-Apr-2007,"['L16WN', 'N14S421']",6496,106074,0.0,0.0,613.77
8,556155,13-Apr-2007,"['2867', 'L16WN']",6474,106053,0.0,0.0,613.77
9,556155,13-Apr-2007,"['L16WN', 'Y2B125', 'E23001']",6279,117139,0.0,0.0,613.77



TASK 3: NETWORK ANALYSIS — CONSOLIDATED SUMMARY
Client–Client graph  : 36,964 nodes | 173,687 directed edges
Member–Client graph  : 37,512 nodes  | 37,662 edges
Reciprocal pairs     : 14,691  (balanced ≥0.9: 3,413)
Circular loops       : 1,809  (infra-linked: 3)
Members routing loop clients : 284 of 548
Highly synced loops  : 1,809


## Stage 1 — Task 4: Explainable Risk Scoring
Produce three scored outputs per the guidelines:
1. **Client-level risk score** — composite of loop participation, reciprocity, synchronization, and infrastructure signals
2. **Member-level risk score** — concentration of high-risk clients routed + volume asymmetry
3. **Evidence strength index** — how many independent signals converge on each entity

In [10]:
import numpy as np

# ─────────────────────────────────────────────────────────────────────────────
# PRE-COMPUTE: Per-client synchronization stats from sync_df
# ─────────────────────────────────────────────────────────────────────────────
import ast

sync_client_records = []
for _, row in sync_df.iterrows():
    try:
        nodes = [str(n).strip() for n in ast.literal_eval(row['Cluster_Nodes'])]
    except Exception:
        continue
    for client in nodes:
        sync_client_records.append({
            'client_id'      : client,
            'Median_Gap_Sec' : row['Median_Gap_Sec'],
            'Min_Gap_Sec'    : row['Min_Gap_Sec'],
        })

sync_client_df = pd.DataFrame(sync_client_records)
if not sync_client_df.empty:
    sync_agg = (
        sync_client_df.groupby('client_id')
        .agg(Avg_Median_Gap=('Median_Gap_Sec','mean'),
             Min_Gap_Ever=('Min_Gap_Sec','min'))
        .reset_index()
    )
else:
    sync_agg = pd.DataFrame(columns=['client_id','Avg_Median_Gap','Min_Gap_Ever'])

# ─────────────────────────────────────────────────────────────────────────────
# ── CLIENT-LEVEL RISK SCORE ──────────────────────────────────────────────────
#
# Five independently measurable signals, each scaled 0–20 → total max = 100
#
# S1 (0-20): Loop Participation intensity
#   = min(Loops_Participated / 10, 1) * 20
# S2 (0-20): Reciprocity Ratio
#   = Reciprocity_Ratio * 20
# S3 (0-20): Volume Weight (log-scaled bottleneck volume)
#   = min(log10(Total_Bottleneck_Vol+1) / log10(500000), 1) * 20
# S4 (0-20): Synchronization (tightness of co-trading)
#   = (1 - min(Avg_Median_Gap, 300) / 300) * 20    [lower gap = higher score]
# S5 (0-20): Infrastructure Link (binary + trader-id bonus)
#   = 10 if Any_Infra_Linked, +10 if shared trader ID specifically
# ─────────────────────────────────────────────────────────────────────────────

# Build master client-level frame from loop_signals + trade_activity + sync + infra
client_scores = loop_signals.copy()          # has scrip_code, client_id + loop signals
client_scores = client_scores.merge(
    trade_activity[['scrip_code','client_id','Total_Buy_Vol','Total_Sell_Vol',
                    'Total_Traded_Vol','Reciprocity_Ratio']],
    on=['scrip_code','client_id'], how='left')
client_scores = client_scores.merge(sync_agg, on='client_id', how='left')
client_scores.fillna({
    'Total_Buy_Vol':0,'Total_Sell_Vol':0,'Total_Traded_Vol':0,
    'Reciprocity_Ratio':0,'Avg_Median_Gap':300,'Min_Gap_Ever':300,
    'Any_Infra_Linked': False,
}, inplace=True)

# Compute shared-trader flag per client from infra_summary
shared_trader_clients = set()
for _, loop_row in optimized_loops_df[optimized_loops_df['Shared_Trader']].iterrows():
    try:
        nodes = [str(n).strip() for n in ast.literal_eval(loop_row['Cluster_Nodes'])]
        shared_trader_clients.update(nodes)
    except Exception:
        pass

client_scores['Shared_Trader_Flag'] = client_scores['client_id'].isin(shared_trader_clients)

# Score components
client_scores['S1_Loop_Intensity'] = (
    (client_scores['Loops_Participated'] / 10).clip(upper=1) * 20
).round(2)

client_scores['S2_Reciprocity'] = (
    client_scores['Reciprocity_Ratio'] * 20
).round(2)

client_scores['S3_Volume_Weight'] = (
    (np.log10(client_scores['Total_Bottleneck_Vol'] + 1) /
     np.log10(500_000)).clip(upper=1) * 20
).round(2)

client_scores['S4_Synchronization'] = (
    (1 - (client_scores['Avg_Median_Gap'].clip(upper=300) / 300)) * 20
).round(2)

client_scores['S5_Infrastructure'] = (
    client_scores['Any_Infra_Linked'].astype(int) * 10 +
    client_scores['Shared_Trader_Flag'].astype(int) * 10
)

client_scores['Client_Risk_Score'] = (
    client_scores[['S1_Loop_Intensity','S2_Reciprocity',
                   'S3_Volume_Weight','S4_Synchronization',
                   'S5_Infrastructure']].sum(axis=1)
).round(2)

# Risk band
def risk_band(score):
    if score >= 75: return 'CRITICAL'
    if score >= 55: return 'HIGH'
    if score >= 35: return 'MEDIUM'
    if score > 0:   return 'LOW'
    return 'NONE'

client_scores['Risk_Band'] = client_scores['Client_Risk_Score'].apply(risk_band)

# Evidence Strength Index — count of signals > threshold
client_scores['Evidence_Strength'] = (
    (client_scores['S1_Loop_Intensity'] >= 8).astype(int) +
    (client_scores['S2_Reciprocity']    >= 14).astype(int) +
    (client_scores['S3_Volume_Weight']  >= 8).astype(int) +
    (client_scores['S4_Synchronization']>= 15).astype(int) +
    (client_scores['S5_Infrastructure'] >= 10).astype(int)
)

# ─────────────────────────────────────────────────────────────────────────────
# ── MEMBER-LEVEL RISK SCORE ───────────────────────────────────────────────────
#
# M1 (0-40): % of clients routed that are HIGH/CRITICAL risk
# M2 (0-30): Total volume through high-risk clients as % of member's total
# M3 (0-30): Number of CRITICAL clients routed (log-scaled)
# ─────────────────────────────────────────────────────────────────────────────

# Map client risk scores back to member_client_map
client_risk_lookup = (
    client_scores.groupby('client_id')['Client_Risk_Score'].max().reset_index()
)
client_risk_lookup['Risk_Band'] = client_risk_lookup['Client_Risk_Score'].apply(risk_band)

mcm = member_client_map.merge(client_risk_lookup, on='client_id', how='left')
mcm.fillna({'Client_Risk_Score': 0}, inplace=True)
mcm['Risk_Band'] = mcm['Client_Risk_Score'].apply(risk_band)

# High-risk client count per member
high_risk_per_member = (
    mcm[mcm['Risk_Band'].isin(['HIGH','CRITICAL'])]
    .groupby('member_code')['client_id'].nunique()
    .reset_index()
    .rename(columns={'client_id':'High_Risk_Clients'})
)
critical_per_member = (
    mcm[mcm['Risk_Band'] == 'CRITICAL']
    .groupby('member_code')['client_id'].nunique()
    .reset_index()
    .rename(columns={'client_id':'Critical_Clients'})
)

member_risk = member_stats.merge(high_risk_per_member, on='member_code', how='left')
member_risk = member_risk.merge(critical_per_member,   on='member_code', how='left')
member_risk.fillna({'High_Risk_Clients': 0, 'Critical_Clients': 0}, inplace=True)

member_risk['Pct_High_Risk_Clients'] = (
    member_risk['High_Risk_Clients'] / member_risk['Distinct_Clients'] * 100
).round(1)

# Volume through high-risk clients
high_risk_clients_set = set(mcm[mcm['Risk_Band'].isin(['HIGH','CRITICAL'])]['client_id'])

buy_hr_vol = (
    df_trades[df_trades['Buy Client Code'].isin(high_risk_clients_set)]
    .groupby('BUY_MEMBER_CODE')['TRADE_QUANTITY'].sum()
    .reset_index()
    .rename(columns={'BUY_MEMBER_CODE':'member_code','TRADE_QUANTITY':'HighRisk_Buy_Vol'})
)
buy_hr_vol['member_code'] = buy_hr_vol['member_code'].astype(str).str.strip()

sell_hr_vol = (
    df_trades[df_trades['Sell Client Code'].isin(high_risk_clients_set)]
    .groupby('SELL_MEMBER_CODE')['TRADE_QUANTITY'].sum()
    .reset_index()
    .rename(columns={'SELL_MEMBER_CODE':'member_code','TRADE_QUANTITY':'HighRisk_Sell_Vol'})
)
sell_hr_vol['member_code'] = sell_hr_vol['member_code'].astype(str).str.strip()

member_risk = member_risk.merge(buy_hr_vol,  on='member_code', how='left')
member_risk = member_risk.merge(sell_hr_vol, on='member_code', how='left')
member_risk.fillna({'HighRisk_Buy_Vol':0,'HighRisk_Sell_Vol':0}, inplace=True)
member_risk['HighRisk_Total_Vol'] = member_risk['HighRisk_Buy_Vol'] + member_risk['HighRisk_Sell_Vol']
member_risk['HighRisk_Vol_Pct'] = (
    member_risk['HighRisk_Total_Vol'] / member_risk['Total_Vol'].replace(0, np.nan) * 100
).fillna(0).round(1)

# Score
member_risk['M1_HighRisk_Client_Pct'] = (member_risk['Pct_High_Risk_Clients'] / 100 * 40).round(2)
member_risk['M2_HighRisk_Vol_Pct']    = (member_risk['HighRisk_Vol_Pct']       / 100 * 30).round(2)
member_risk['M3_Critical_Clients']    = (
    (np.log1p(member_risk['Critical_Clients']) / np.log1p(20)).clip(upper=1) * 30
).round(2)
member_risk['Member_Risk_Score'] = (
    member_risk[['M1_HighRisk_Client_Pct','M2_HighRisk_Vol_Pct','M3_Critical_Clients']].sum(axis=1)
).round(2)
member_risk['Member_Risk_Band'] = member_risk['Member_Risk_Score'].apply(risk_band)

# ─────────────────────────────────────────────────────────────────────────────
# PRINT RESULTS
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 65)
print("TASK 4A: CLIENT-LEVEL RISK SCORES")
print("=" * 65)
print("Risk band distribution:")
print(client_scores['Risk_Band'].value_counts().to_string())
print(f"\nMean client risk score : {client_scores['Client_Risk_Score'].mean():.1f}")
print(f"Max client risk score  : {client_scores['Client_Risk_Score'].max():.1f}")

print("\nTop 25 highest-risk clients:")
top_clients = client_scores.sort_values('Client_Risk_Score', ascending=False).head(25)
display(top_clients[['scrip_code','client_id','Client_Risk_Score','Risk_Band',
                      'S1_Loop_Intensity','S2_Reciprocity','S3_Volume_Weight',
                      'S4_Synchronization','S5_Infrastructure',
                      'Evidence_Strength']].reset_index(drop=True))

print("\n" + "=" * 65)
print("TASK 4B: MEMBER-LEVEL RISK SCORES")
print("=" * 65)
print("Risk band distribution:")
print(member_risk['Member_Risk_Band'].value_counts().to_string())

print("\nTop 20 highest-risk members:")
top_members = member_risk.sort_values('Member_Risk_Score', ascending=False).head(20)
display(top_members[['member_code','Distinct_Clients','High_Risk_Clients','Critical_Clients',
                      'Pct_High_Risk_Clients','HighRisk_Vol_Pct',
                      'Member_Risk_Score','Member_Risk_Band']].reset_index(drop=True))

print("\n" + "=" * 65)
print("TASK 4C: EVIDENCE STRENGTH INDEX")
print("=" * 65)
print("Distribution of converging signals per client (0=no evidence, 5=all signals):")
print(client_scores['Evidence_Strength'].value_counts().sort_index().to_string())

strong_evidence = client_scores[client_scores['Evidence_Strength'] >= 4]
print(f"\nClients with ≥4 converging signals: {len(strong_evidence)}")
display(strong_evidence[['scrip_code','client_id','Client_Risk_Score','Risk_Band',
                          'Evidence_Strength',
                          'S1_Loop_Intensity','S2_Reciprocity','S3_Volume_Weight',
                          'S4_Synchronization','S5_Infrastructure']].sort_values(
    'Client_Risk_Score', ascending=False).reset_index(drop=True))

# Save scored tables for use in Tasks 5 and later stages
client_scores_final  = client_scores.copy()
member_scores_final  = member_risk.copy()


TASK 4A: CLIENT-LEVEL RISK SCORES
Risk band distribution:
Risk_Band
MEDIUM      576
HIGH        269
CRITICAL     29

Mean client risk score : 55.6
Max client risk score  : 99.8

Top 25 highest-risk clients:


,scrip_code,client_id,Client_Risk_Score,Risk_Band,S1_Loop_Intensity,S2_Reciprocity,S3_Volume_Weight,S4_Synchronization,S5_Infrastructure,Evidence_Strength
0,556155,L16WN,99.84,CRITICAL,20.0,19.84,20.00,20.0,20,5
1,555835,L16WN,84.82,CRITICAL,20.0,19.59,15.23,20.0,10,5
2,556155,K17329,79.42,CRITICAL,20.0,20.00,19.42,20.0,0,4
3,556155,E23001,78.87,CRITICAL,20.0,19.86,19.01,20.0,0,4
4,556155,91N89,78.42,CRITICAL,6.0,20.00,12.42,20.0,20,4
5,556155,Y2B125,78.39,CRITICAL,20.0,20.00,18.39,20.0,0,4
6,556155,7120,76.66,CRITICAL,20.0,19.59,17.07,20.0,0,4
7,556155,G21119,76.56,CRITICAL,20.0,20.00,16.56,20.0,0,4
8,556155,N14G230,76.46,CRITICAL,20.0,19.36,17.10,20.0,0,4
9,556155,X3SL,76.39,CRITICAL,20.0,20.00,16.39,20.0,0,4



TASK 4B: MEMBER-LEVEL RISK SCORES
Risk band distribution:
Member_Risk_Band
NONE        335
LOW         142
MEDIUM       38
CRITICAL     22
HIGH         11

Top 20 highest-risk members:


,member_code,Distinct_Clients,High_Risk_Clients,Critical_Clients,Pct_High_Risk_Clients,HighRisk_Vol_Pct,Member_Risk_Score,Member_Risk_Band
0,734,1,1.0,1.0,100.0,100.0,76.83,CRITICAL
1,3207,1,1.0,1.0,100.0,100.0,76.83,CRITICAL
2,522,1,1.0,1.0,100.0,100.0,76.83,CRITICAL
3,543,1,1.0,1.0,100.0,100.0,76.83,CRITICAL
4,636,1,1.0,1.0,100.0,100.0,76.83,CRITICAL
5,630,1,1.0,1.0,100.0,100.0,76.83,CRITICAL
6,669,1,1.0,1.0,100.0,100.0,76.83,CRITICAL
7,221,1,1.0,1.0,100.0,100.0,76.83,CRITICAL
8,156,1,1.0,1.0,100.0,100.0,76.83,CRITICAL
9,43,1,1.0,1.0,100.0,100.0,76.83,CRITICAL



TASK 4C: EVIDENCE STRENGTH INDEX
Distribution of converging signals per client (0=no evidence, 5=all signals):
Evidence_Strength
2     14
3    712
4    146
5      2

Clients with ≥4 converging signals: 148


,scrip_code,client_id,Client_Risk_Score,Risk_Band,Evidence_Strength,S1_Loop_Intensity,S2_Reciprocity,S3_Volume_Weight,S4_Synchronization,S5_Infrastructure
0,556155,L16WN,99.84,CRITICAL,5,20.0,19.84,20.00,20.0,20
1,555835,L16WN,84.82,CRITICAL,5,20.0,19.59,15.23,20.0,10
2,556155,K17329,79.42,CRITICAL,4,20.0,20.00,19.42,20.0,0
3,556155,E23001,78.87,CRITICAL,4,20.0,19.86,19.01,20.0,0
4,556155,91N89,78.42,CRITICAL,4,6.0,20.00,12.42,20.0,20
...,...,...,...,...,...,...,...,...,...,...
143,556155,5088,60.07,HIGH,4,8.0,20.00,12.07,20.0,0
144,556155,29C005,59.99,HIGH,4,8.0,20.00,11.99,20.0,0
145,556155,P11OLK47803,59.90,HIGH,4,8.0,20.00,11.90,20.0,0
146,556155,67B1,59.69,HIGH,4,8.0,20.00,11.69,20.0,0


## Stage 1 — Task 5: Identify Additional High-Risk Entities
Expand detection **beyond the original suspect list** using four methods as mandated:
1. **Network expansion** — clients 1-hop from confirmed high-risk nodes in `Client_DG`
2. **Motif participation** — clients in loops not on the suspect list at all
3. **Synchrony similarity** — unlisted clients whose trade timing mirrors known manipulators
4. **Counterparty concentration** — unlisted clients whose trading is dominated by confirmed high-risk counterparties

In [11]:
import ast
import numpy as np

# Known suspect client IDs (original list)
known_suspects = set(suspects_df['client_id'].astype(str).str.strip().unique())

# CRITICAL/HIGH clients identified in Task 4
confirmed_high_risk = set(
    client_scores_final[client_scores_final['Risk_Band'].isin(['CRITICAL','HIGH'])]['client_id']
    .astype(str).str.strip().unique()
)

# All clients in detected loops (regardless of suspect list)
loop_clients_all = set(loops_full['client_id'].astype(str).str.strip().unique())

print(f"Known suspects         : {len(known_suspects):,}")
print(f"Confirmed high-risk    : {len(confirmed_high_risk):,}")
print(f"All loop participants  : {len(loop_clients_all):,}")
print(f"Loop clients NOT on suspect list: "
      f"{len(loop_clients_all - known_suspects):,}")

# ─────────────────────────────────────────────────────────────────────────────
# METHOD 1: NETWORK EXPANSION
# Follow directed edges from confirmed high-risk nodes in Client_DG.
# 1-hop neighbours who are NOT on the suspect list = newly exposed entities.
# ─────────────────────────────────────────────────────────────────────────────

network_expanded = set()
for node in confirmed_high_risk:
    if node in Client_DG:
        # Successors = clients that bought FROM this high-risk seller
        network_expanded.update(Client_DG.successors(node))
        # Predecessors = clients that sold TO this high-risk buyer
        network_expanded.update(Client_DG.predecessors(node))

network_new = network_expanded - known_suspects - confirmed_high_risk
print(f"\nMETHOD 1 — Network Expansion: {len(network_new):,} new entities exposed as 1-hop neighbours")

# Score the expanded neighbours: edge weight to/from high-risk nodes
expansion_records = []
for client in network_new:
    client = str(client)
    vol_to_hr   = sum(Client_DG[client][nb]['weight']
                      for nb in Client_DG.successors(client)
                      if nb in confirmed_high_risk) if client in Client_DG else 0
    vol_from_hr = sum(Client_DG[nb][client]['weight']
                      for nb in Client_DG.predecessors(client)
                      if nb in confirmed_high_risk) if client in Client_DG else 0
    expansion_records.append({
        'client_id'        : client,
        'Vol_to_HighRisk'  : vol_to_hr,
        'Vol_from_HighRisk': vol_from_hr,
        'Total_HR_Exposure': vol_to_hr + vol_from_hr,
        'Detection_Method' : 'Network Expansion',
    })

expansion_df = pd.DataFrame(expansion_records).sort_values('Total_HR_Exposure', ascending=False)

# ─────────────────────────────────────────────────────────────────────────────
# METHOD 2: MOTIF PARTICIPATION
# Clients appearing in detected loops who are NOT on the original suspect list.
# ─────────────────────────────────────────────────────────────────────────────

motif_new = loop_clients_all - known_suspects
print(f"METHOD 2 — Motif Participation: {len(motif_new):,} loop participants not on suspect list")

# Their loop stats
motif_stats = (
    loops_full[loops_full['client_id'].isin(motif_new)]
    .groupby('client_id')
    .agg(
        Loops_In           = ('Bottleneck_Volume', 'count'),
        Total_Bottleneck   = ('Bottleneck_Volume', 'sum'),
        Max_Bottleneck     = ('Bottleneck_Volume', 'max'),
        Distinct_Scrips    = ('SCRIP_CODE', 'nunique'),
        Distinct_Dates     = ('TRADE_DATE', 'nunique'),
    )
    .reset_index()
    .sort_values('Total_Bottleneck', ascending=False)
)
motif_stats['Detection_Method'] = 'Motif Participation'

# ─────────────────────────────────────────────────────────────────────────────
# METHOD 3: SYNCHRONY SIMILARITY
# Unlisted clients whose median trade gap on high-activity scrip-dates
# closely matches the confirmed manipulators' timing profile.
# Threshold: median gap within ±20% of the manipulators' median gap.
# ─────────────────────────────────────────────────────────────────────────────

# Reference: median gap of confirmed high-risk clients
hr_sync = sync_agg[sync_agg['client_id'].isin(confirmed_high_risk)]
if not hr_sync.empty:
    ref_gap = hr_sync['Avg_Median_Gap'].median()
    gap_lo  = ref_gap * 0.80
    gap_hi  = ref_gap * 1.20
else:
    ref_gap = gap_lo = gap_hi = None

# Compute per-client sync stats for ALL clients (not just suspects)
all_sync_records = []
for _, row in sync_df.iterrows():
    try:
        nodes = [str(n).strip() for n in ast.literal_eval(row['Cluster_Nodes'])]
    except Exception:
        continue
    for client in nodes:
        all_sync_records.append({
            'client_id'      : client,
            'Median_Gap_Sec' : row['Median_Gap_Sec'],
        })

all_sync_df = pd.DataFrame(all_sync_records)
if not all_sync_df.empty:
    all_sync_agg = all_sync_df.groupby('client_id')['Median_Gap_Sec'].mean().reset_index()
    all_sync_agg.columns = ['client_id', 'Avg_Gap']
else:
    all_sync_agg = pd.DataFrame(columns=['client_id','Avg_Gap'])

if ref_gap is not None:
    sync_similar = all_sync_agg[
        (all_sync_agg['Avg_Gap'] >= gap_lo) &
        (all_sync_agg['Avg_Gap'] <= gap_hi) &
        (~all_sync_agg['client_id'].isin(known_suspects))
    ].copy()
    sync_similar['Ref_Gap']          = ref_gap
    sync_similar['Detection_Method'] = 'Synchrony Similarity'
    print(f"METHOD 3 — Synchrony Similarity: {len(sync_similar):,} unlisted clients "
          f"with timing matching confirmed manipulators (ref gap={ref_gap:.1f}s ±20%)")
else:
    sync_similar = pd.DataFrame(columns=['client_id','Avg_Gap','Ref_Gap','Detection_Method'])
    print("METHOD 3 — Synchrony Similarity: No reference gap available (no high-risk sync data)")

# ─────────────────────────────────────────────────────────────────────────────
# METHOD 4: COUNTERPARTY CONCENTRATION
# Unlisted clients where >50% of their total traded volume is with confirmed
# high-risk counterparties — suggests they are witting participants.
# ─────────────────────────────────────────────────────────────────────────────

# Total volume each client trades with high-risk counterparties
buy_hr  = df_trades[df_trades['Sell Client Code'].isin(confirmed_high_risk)][
    ['Buy Client Code','TRADE_QUANTITY']].rename(
    columns={'Buy Client Code':'client_id','TRADE_QUANTITY':'HR_Buy_Vol'})
sell_hr = df_trades[df_trades['Buy Client Code'].isin(confirmed_high_risk)][
    ['Sell Client Code','TRADE_QUANTITY']].rename(
    columns={'Sell Client Code':'client_id','TRADE_QUANTITY':'HR_Sell_Vol'})

hr_exposure = (
    buy_hr.groupby('client_id')['HR_Buy_Vol'].sum()
    .reset_index()
    .merge(sell_hr.groupby('client_id')['HR_Sell_Vol'].sum().reset_index(),
           on='client_id', how='outer')
    .fillna(0)
)
hr_exposure['Total_HR_Vol'] = hr_exposure['HR_Buy_Vol'] + hr_exposure['HR_Sell_Vol']

# Total volume for each client overall
total_vol_per_client = (
    trade_activity.groupby('client_id')['Total_Traded_Vol'].sum().reset_index()
)
hr_exposure = hr_exposure.merge(total_vol_per_client, on='client_id', how='left').fillna(0)
hr_exposure['HR_Vol_Pct'] = (
    hr_exposure['Total_HR_Vol'] / hr_exposure['Total_Traded_Vol'].replace(0, np.nan) * 100
).fillna(0).round(1)

# Filter: >50% exposure, not already on suspect list, minimum volume
cp_concentration = hr_exposure[
    (hr_exposure['HR_Vol_Pct'] >= 50) &
    (hr_exposure['Total_Traded_Vol'] >= 10_000) &
    (~hr_exposure['client_id'].isin(known_suspects))
].copy()
cp_concentration['Detection_Method'] = 'Counterparty Concentration'
cp_concentration = cp_concentration.sort_values('Total_HR_Vol', ascending=False)

print(f"METHOD 4 — Counterparty Concentration: {len(cp_concentration):,} unlisted clients "
      f"with >50% volume exposure to confirmed high-risk clients")

# ─────────────────────────────────────────────────────────────────────────────
# CONSOLIDATE — MASTER NEW ENTITIES TABLE
# Union all four method outputs, de-duplicate, score via existing scoring
# ─────────────────────────────────────────────────────────────────────────────

new_entity_ids = set()
new_entity_ids.update(expansion_df.head(200)['client_id'].tolist())
new_entity_ids.update(motif_stats.head(200)['client_id'].tolist())
new_entity_ids.update(sync_similar['client_id'].tolist() if not sync_similar.empty else [])
new_entity_ids.update(cp_concentration.head(200)['client_id'].tolist())

# Remove those already on the suspect list
new_entity_ids -= known_suspects

print(f"\nTotal new unique high-risk entities identified: {len(new_entity_ids):,}")

# Apply same risk scoring logic to new entities
new_scores = loop_signals[loop_signals['client_id'].isin(new_entity_ids)].copy() if not loop_signals.empty else pd.DataFrame()

if not new_scores.empty:
    new_scores = new_scores.merge(
        trade_activity[['scrip_code','client_id','Total_Buy_Vol','Total_Sell_Vol',
                        'Total_Traded_Vol','Reciprocity_Ratio']],
        on=['scrip_code','client_id'], how='left')
    new_scores = new_scores.merge(sync_agg, on='client_id', how='left')
    new_scores.fillna({'Total_Buy_Vol':0,'Total_Sell_Vol':0,'Total_Traded_Vol':0,
                       'Reciprocity_Ratio':0,'Avg_Median_Gap':300,'Any_Infra_Linked':False}, inplace=True)
    new_scores['Shared_Trader_Flag'] = new_scores['client_id'].isin(shared_trader_clients)
    new_scores['S1_Loop_Intensity']  = ((new_scores['Loops_Participated']/10).clip(upper=1)*20).round(2)
    new_scores['S2_Reciprocity']     = (new_scores['Reciprocity_Ratio']*20).round(2)
    new_scores['S3_Volume_Weight']   = ((np.log10(new_scores['Total_Bottleneck_Vol']+1)/np.log10(500_000)).clip(upper=1)*20).round(2)
    new_scores['S4_Synchronization'] = ((1-(new_scores['Avg_Median_Gap'].clip(upper=300)/300))*20).round(2)
    new_scores['S5_Infrastructure']  = (new_scores['Any_Infra_Linked'].astype(int)*10 + new_scores['Shared_Trader_Flag'].astype(int)*10)
    new_scores['Client_Risk_Score']  = new_scores[['S1_Loop_Intensity','S2_Reciprocity','S3_Volume_Weight','S4_Synchronization','S5_Infrastructure']].sum(axis=1).round(2)
    new_scores['Risk_Band']          = new_scores['Client_Risk_Score'].apply(risk_band)
    new_scores['Source']             = 'Newly Identified'

# Tag detection method(s) for each new entity
method_tags = {}
for _, r in expansion_df.iterrows():
    method_tags.setdefault(r['client_id'], []).append('Network Expansion')
for _, r in motif_stats.iterrows():
    method_tags.setdefault(r['client_id'], []).append('Motif Participation')
if not sync_similar.empty:
    for _, r in sync_similar.iterrows():
        method_tags.setdefault(r['client_id'], []).append('Synchrony Similarity')
for _, r in cp_concentration.iterrows():
    method_tags.setdefault(r['client_id'], []).append('Counterparty Concentration')

if not new_scores.empty:
    new_scores['Detection_Methods'] = new_scores['client_id'].map(
        lambda c: ' | '.join(method_tags.get(c, ['Unknown'])))

# ─────────────────────────────────────────────────────────────────────────────
# PRINT RESULTS
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("TASK 5: ADDITIONAL HIGH-RISK ENTITIES — SUMMARY")
print("=" * 65)
print(f"Method 1 — Network Expansion      : {len(expansion_df):>5,} candidates")
print(f"Method 2 — Motif Participation     : {len(motif_stats):>5,} candidates")
print(f"Method 3 — Synchrony Similarity    : {len(sync_similar):>5,} candidates")
print(f"Method 4 — Counterparty Conc.      : {len(cp_concentration):>5,} candidates")
print(f"Unique new entities (union)        : {len(new_entity_ids):>5,}")

if not new_scores.empty:
    print(f"\nNew entities with risk scores:")
    print(new_scores['Risk_Band'].value_counts().to_string())
    print(f"\nTop 30 highest-risk NEW entities (not on original suspect list):")
    display(new_scores.sort_values('Client_Risk_Score', ascending=False).head(30)[
        ['scrip_code','client_id','Client_Risk_Score','Risk_Band',
         'S1_Loop_Intensity','S2_Reciprocity','S3_Volume_Weight',
         'S4_Synchronization','Detection_Methods']].reset_index(drop=True))

print("\nTop 20 — METHOD 2: Loop participants NOT on suspect list:")
display(motif_stats.head(20).reset_index(drop=True))

print("\nTop 20 — METHOD 4: Counterparty concentration (unlisted clients):")
display(cp_concentration.head(20)[['client_id','HR_Buy_Vol','HR_Sell_Vol',
                                    'Total_HR_Vol','Total_Traded_Vol',
                                    'HR_Vol_Pct','Detection_Method']].reset_index(drop=True))

# Save for Stage 2
new_entities_final = new_scores.copy() if not new_scores.empty else pd.DataFrame()


Known suspects         : 1,765
Confirmed high-risk    : 294
All loop participants  : 868
Loop clients NOT on suspect list: 437

METHOD 1 — Network Expansion: 29,916 new entities exposed as 1-hop neighbours
METHOD 2 — Motif Participation: 437 loop participants not on suspect list
METHOD 3 — Synchrony Similarity: 420 unlisted clients with timing matching confirmed manipulators (ref gap=0.0s ±20%)
METHOD 4 — Counterparty Concentration: 68 unlisted clients with >50% volume exposure to confirmed high-risk clients

Total new unique high-risk entities identified: 493

TASK 5: ADDITIONAL HIGH-RISK ENTITIES — SUMMARY
Method 1 — Network Expansion      : 29,916 candidates
Method 2 — Motif Participation     :   437 candidates
Method 3 — Synchrony Similarity    :   420 candidates
Method 4 — Counterparty Conc.      :    68 candidates
Unique new entities (union)        :   493

New entities with risk scores:
Risk_Band
MEDIUM      331
HIGH         97
CRITICAL      3

Top 30 highest-risk NEW entities (

,scrip_code,client_id,Client_Risk_Score,Risk_Band,S1_Loop_Intensity,S2_Reciprocity,S3_Volume_Weight,S4_Synchronization,Detection_Methods
0,556155,91N89,78.42,CRITICAL,6.0,20.00,12.42,20.0,Motif Participation | Synchrony Similarity | C...
1,556155,Y2AM199,75.11,CRITICAL,20.0,20.00,15.11,20.0,Motif Participation | Counterparty Concentration
2,556155,H20044,75.07,CRITICAL,20.0,19.77,15.30,20.0,Motif Participation | Synchrony Similarity | C...
3,556155,63028,74.80,HIGH,20.0,20.00,14.80,20.0,Motif Participation | Synchrony Similarity | C...
4,556155,U62,74.48,HIGH,20.0,19.97,14.51,20.0,Motif Participation | Synchrony Similarity | C...
5,556155,U64,74.44,HIGH,20.0,19.96,14.48,20.0,Motif Participation | Counterparty Concentration
6,556155,W4B999,74.17,HIGH,20.0,19.98,14.19,20.0,Motif Participation | Counterparty Concentration
7,556155,87933,74.11,HIGH,20.0,20.00,14.11,20.0,Motif Participation | Counterparty Concentration
8,556155,B26S5001,74.11,HIGH,20.0,20.00,14.11,20.0,Motif Participation | Synchrony Similarity | C...
9,556155,Y2W7030,74.01,HIGH,20.0,20.00,14.01,20.0,Motif Participation | Synchrony Similarity | C...



Top 20 — METHOD 2: Loop participants NOT on suspect list:


,client_id,Loops_In,Total_Bottleneck,Max_Bottleneck,Distinct_Scrips,Distinct_Dates,Detection_Method
0,H20044,11,22858,9323,1,1,Motif Participation
1,Y2AM199,16,20183,7159,1,1,Motif Participation
2,63028,12,16488,9331,1,1,Motif Participation
3,U62,12,13653,5000,1,1,Motif Participation
4,U64,10,13334,2716,1,1,Motif Participation
5,W4B999,11,11062,3865,1,1,Motif Participation
6,B26S5001,11,10499,1850,1,1,Motif Participation
7,87933,13,10497,3431,1,1,Motif Participation
8,Y2H714,9,10352,4918,1,1,Motif Participation
9,Y2W7030,13,9817,3518,1,1,Motif Participation



Top 20 — METHOD 4: Counterparty concentration (unlisted clients):


,client_id,HR_Buy_Vol,HR_Sell_Vol,Total_HR_Vol,Total_Traded_Vol,HR_Vol_Pct,Detection_Method
0,S8DFCMF,101973.0,0.0,101973.0,180000.0,56.7,Counterparty Concentration
1,T7002,16515.0,19121.0,35636.0,38343.0,92.9,Counterparty Concentration
2,H20044,20078.0,14129.0,34207.0,43150.0,79.3,Counterparty Concentration
3,Y2AM199,16952.0,14530.0,31482.0,39442.0,79.8,Counterparty Concentration
4,63028,13774.0,13781.0,27555.0,34800.0,79.2,Counterparty Concentration
5,91C75,8901.0,10686.0,19587.0,33470.0,58.5,Counterparty Concentration
6,U62,9500.0,9123.0,18623.0,23020.0,80.9,Counterparty Concentration
7,87933,10736.0,7841.0,18577.0,28382.0,65.5,Counterparty Concentration
8,Y2H714,8844.0,9409.0,18253.0,24000.0,76.1,Counterparty Concentration
9,X3Y008,10483.0,7232.0,17715.0,30494.0,58.1,Counterparty Concentration
